In [ ]:
import ftplib
from pathlib import Path

DIR_SAIDA = Path("dados_tb")
DIR_SAIDA.mkdir(exist_ok=True)
ARQUIVOS = ["TUBEBR20.dbc","TUBEBR21.dbc", "TUBEBR22.dbc", "TUBEBR23.dbc", "TUBEBR24.dbc"]

ftp = ftplib.FTP("ftp.datasus.gov.br")
ftp.login()
ftp.cwd("/dissemin/publicos/SINAN/DADOS/PRELIM/")

for arquivo in ARQUIVOS:
    dbc_path = DIR_SAIDA / arquivo
    print(f"Baixando {arquivo}...")
    with open(dbc_path, "wb") as f:
        ftp.retrbinary(f"RETR {arquivo}", f.write)
    print(f"  {dbc_path.stat().st_size:,} bytes")

ftp.quit()
print("Download concluído!")

In [ ]:
!pip install pyreaddbc
!pip install dbfread

import pyreaddbc
import pandas as pd
from dbfread import DBF
from pathlib import Path

DIR_SAIDA = Path("dados_tb")
ARQUIVOS = ["TUBEBR20.dbc", "TUBEBR21.dbc", "TUBEBR22.dbc", "TUBEBR23.dbc", "TUBEBR24.dbc"]

for arquivo in ARQUIVOS:
    dbc_path = str(DIR_SAIDA / arquivo)
    dbf_path = dbc_path.replace(".dbc", ".dbf")
    csv_path = dbc_path.replace(".dbc", ".csv")

    print(f"Convertendo {arquivo}...")
    pyreaddbc.dbc2dbf(dbc_path, dbf_path)

    # Confirma que o DBF foi gerado
    if not Path(dbf_path).exists() or Path(dbf_path).stat().st_size == 0:
        print(f"ERRO: DBF não gerado para {arquivo}, pulando.")
        continue

    df = pd.DataFrame(iter(DBF(dbf_path, encoding="latin-1")))
    df.to_csv(csv_path, index=False, encoding="utf-8")
    print(f"Salvo: {csv_path} — {len(df):,} linhas, {df.shape[1]} colunas\n")

print("Concluído!")

In [ ]:
"""
TB_abandono_visualizacoes.py
====================================
Pré-processamento dos registros do SINAN-TB referentes ao estado de Pernambuco
(2020–2024): leitura dos CSVs anuais, filtragem por UF de residência,
classificação de desfechos, atribuição de indicadores de vulnerabilidade social
e geração das figuras para o manuscrito.

Fonte dos dados: SINAN-TB via PySUS (TUBEBR20–TUBEBR24).
"""

# ─────────────────────────────────────────
# DEPENDÊNCIAS
# ─────────────────────────────────────────

import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # backend sem GUI; necessário para salvar figuras fora de ambiente interativo
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import chi2_contingency
import statsmodels.api as sm

warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════════════════════════════
# BLOCO 1 — Leitura e consolidação dos registros anuais (2020–2024)
# ═══════════════════════════════════════════════════════════════════════════════
# Cada arquivo CSV corresponde a um ano epidemiológico do SINAN-TB exportado
# via PySUS. Os DataFrames são concatenados em um único objeto para que o
# processamento subsequente cubra todo o período de análise.

print("Lendo dados de TB 2020–2024...")

DIR_SAIDA = Path("dados_tb")
ARQUIVOS  = ["TUBEBR20.csv", "TUBEBR21.csv", "TUBEBR22.csv", "TUBEBR23.csv", "TUBEBR24.csv"]

dfs = []
for arquivo in ARQUIVOS:
    path = DIR_SAIDA / arquivo
    df   = pd.read_csv(path, encoding="utf-8", low_memory=False)
    dfs.append(df)
    print(f"  {arquivo}: {len(df):,} linhas")

df_total = pd.concat(dfs, ignore_index=True)
print(f"\n[BLOCO 1] Total de registros (sem filtro): {len(df_total):,}")

# ═══════════════════════════════════════════════════════════════════════════════
# BLOCO 2 — Filtro por UF de residência: Pernambuco (código IBGE = 26)
# ═══════════════════════════════════════════════════════════════════════════════
# A variável SG_UF representa a UF de residência do paciente, não de notificação,
# o que evita o viés de casos notificados em PE por pessoas domiciliadas em
# outros estados.

mask_pe = df_total['SG_UF'].astype(str).str.strip() == '26'

df_pe = df_total[mask_pe].copy()

print(f"\n[BLOCO 2] Registros residentes em PE:    {mask_pe.sum():,}")
print(f"\n          Distribuição por ano:")
print(df_pe['NU_ANO'].value_counts().sort_index().to_string())

# ═══════════════════════════════════════════════════════════════════════════════
# BLOCO 3 — Indicadores de vulnerabilidade social
# ═══════════════════════════════════════════════════════════════════════════════
# Cada indicador é derivado de variáveis categóricas do SINAN-TB. A função
# abaixo converte numericamente o campo e verifica pertencimento a um conjunto
# de valores predefinidos, retornando uma série booleana. Campos ausentes na
# base resultam em False para toda a série, sem interromper o processamento.

def flag(df, col, valores):
    """Retorna True para linhas cujo campo `col` assume um dos `valores` esperados.

    Colunas ausentes na base resultam em False para toda a série, preservando
    a integridade do DataFrame independentemente do ano de competência.
    """
    if col not in df.columns:
        return pd.Series(False, index=df.index)
    serie = pd.to_numeric(df[col], errors='coerce')
    return serie.isin([float(v) for v in valores])

df_pe['RE_ESCOL']      = flag(df_pe, 'CS_ESCOL_N', [0, 1, 2, 3, 4, 5])   # até fund. II incompleto
df_pe['RE_RACA']       = flag(df_pe, 'CS_RACA',    [2, 4, 5])              # negros, pardos e indígenas
df_pe['RE_BENEF_GOV']  = flag(df_pe, 'BENEF_GOV',  [1])
df_pe['RE_POP_LIBER']  = flag(df_pe, 'POP_LIBER',  [1])
df_pe['RE_POP_RUA']    = flag(df_pe, 'POP_RUA',    [1])
df_pe['RE_POP_IMIG']   = flag(df_pe, 'POP_IMIG',   [1])
df_pe['RE_AGRAVALCOO'] = flag(df_pe, 'AGRAVALCOO', [1])
df_pe['RE_AGRAVDROGA'] = flag(df_pe, 'AGRAVDROGA', [1])

fatores = [
    ('RE_ESCOL',      'Baixa escolaridade (até fund. incompleto)'),
    ('RE_RACA',       'Pessoas pretas, pardas ou indígenas'),
    ('RE_BENEF_GOV',  'Beneficiários de programa social'),
    ('RE_POP_LIBER',  'Privados de liberdade'),
    ('RE_POP_RUA',    'Em situação de rua'),
    ('RE_POP_IMIG',   'Imigrantes'),
    ('RE_AGRAVALCOO', 'Alcoolismo'),
    ('RE_AGRAVDROGA', 'Uso de drogas'),
]

df_pe['tem_fator'] = df_pe[[col for col, _ in fatores]].any(axis=1)

print(f"\n[BLOCO 3] Registros com ao menos 1 fator de vulnerabilidade: {df_pe['tem_fator'].sum():,}")
print(f"\nDetalhamento por fator:")
print(f"{'Fator':<45} {'N':>7}  {'%':>6}")
print("-" * 62)
for col, label in fatores:
    n   = int(df_pe[col].sum())
    pct = df_pe[col].mean() * 100
    print(f"{label:<45} {n:>7,}  {pct:>5.1f}%")

# ═══════════════════════════════════════════════════════════════════════════════
# BLOCO 4 — Exportação da base filtrada em formato CSV
# ═══════════════════════════════════════════════════════════════════════════════

df_pe.to_csv('TB_PYSUS_2020_2024_PE_todos.csv', index=False)
print(f"\n[EXPORT] {len(df_pe):,} registros → TB_PYSUS_2020_2024_PE_todos.csv")

# ═══════════════════════════════════════════════════════════════════════════════
# BLOCO 5 — Preparação das variáveis para visualização
# ═══════════════════════════════════════════════════════════════════════════════

df = pd.read_csv('TB_PYSUS_2020_2024_PE_todos.csv')

# Conversão das datas de início e encerramento do tratamento
for col in ['DT_INIC_TR', 'DT_ENCERRA']:
    df[col] = pd.to_datetime(
        df[col].astype(str).str.split('.').str[0],
        errors='coerce', format='%Y%m%d'
    )
df['DT_NOTIFIC'] = pd.to_datetime(df['DT_NOTIFIC'], errors='coerce')

# A variável NU_IDADE_N codifica a unidade etária no primeiro dígito:
# 4 = anos completos; 2 = meses. Demais unidades (dias, horas) são descartadas.
def decode_idade(x):
    if pd.isna(x):
        return np.nan
    s = str(int(x))
    if s.startswith('4'):
        return int(s[1:])
    if s.startswith('2'):
        return int(s[1:]) / 12
    return np.nan

df['IDADE_ANOS']   = df['NU_IDADE_N'].apply(decode_idade)
df['FAIXA_ETARIA'] = pd.cut(
    df['IDADE_ANOS'],
    bins=[0, 20, 35, 50, 65, 120],
    labels=['<20', '20–34', '35–49', '50–64', '65+']
)

# Comorbidades associadas ao desfecho do tratamento
df['HIV_POS']   = pd.to_numeric(df['AGRAVAIDS'],  errors='coerce') == 1.0
df['DIAB_POS']  = pd.to_numeric(df['AGRAVDIABE'], errors='coerce') == 1.0
df['TABAC_POS'] = pd.to_numeric(df['AGRAVTABAC'], errors='coerce') == 1.0

# Modalidade de acompanhamento: tratamento diretamente observado (TDO)
df['TDO_LABEL'] = pd.to_numeric(df['TRATSUP_AT'], errors='coerce').map({
    1.0: 'TDO supervisionado',
    2.0: 'Auto-administrado',
    9.0: 'Ignorado'
}).fillna('Sem registro')

# Agrupamento dos desfechos clínicos conforme critérios do PNCT.
# Códigos 3 e 4 correspondem a óbito por TB e por outras causas, respectivamente.
# Códigos 2 e 7 representam abandono primário e abandono pós-retratamento.
DESFECHO_MAP = {
    1.0: 'Cura',    2.0: 'Abandono', 3.0: 'Óbito',
    4.0: 'Óbito',   5.0: 'Outros',   7.0: 'Abandono',
    8.0: 'Outros',  10.0: 'Outros'
}
df['DESFECHO_GRUPO'] = pd.to_numeric(df['SITUA_ENCE'], errors='coerce').map(DESFECHO_MAP)

# Raça/cor: pretos (2) e pardos (4) agrupados como "Negra" segundo a PNAD/IBGE
df['RACA_GRUPO'] = pd.to_numeric(df['CS_RACA'], errors='coerce').map({
    1.0: 'Branca', 2.0: 'Negra',    3.0: 'Amarela',
    4.0: 'Negra',  5.0: 'Indígena', 9.0: 'Ignorado'
})

# Escolaridade agrupada em três níveis para reduzir esparsidade nas células
df['ESCOL_GRUPO'] = pd.to_numeric(df['CS_ESCOL_N'], errors='coerce').map({
    0.0: 'Baixa', 1.0: 'Baixa', 2.0: 'Baixa', 3.0: 'Baixa',
    4.0: 'Média', 5.0: 'Média', 6.0: 'Média',
    7.0: 'Alta',  8.0: 'Alta',
    9.0: 'Ignorado', 10.0: 'N/A'
})

df['SEXO_LABEL'] = df['CS_SEXO'].map({'M': 'Masculino', 'F': 'Feminino'})

# Re-atribuição dos indicadores de vulnerabilidade na base de visualização.
# Necessário porque o df foi relido do CSV e as colunas booleanas precisam ser recriadas.
def flag_viz(df, col, valores):
    if col not in df.columns:
        return pd.Series(False, index=df.index)
    return pd.to_numeric(df[col], errors='coerce').isin([float(v) for v in valores])

for col_orig, col_re in [
    ('AGRAVALCOO', 'RE_AGRAVALCOO'),
    ('AGRAVDROGA', 'RE_AGRAVDROGA'),
    ('POP_RUA',    'RE_POP_RUA'),
    ('BENEF_GOV',  'RE_BENEF_GOV'),
    ('POP_LIBER',  'RE_POP_LIBER'),
]:
    df[col_re] = flag_viz(df, col_orig, [1])

GRUPOS  = ['Cura', 'Abandono', 'Óbito']
PALETTE = {'Cura': '#2ecc71', 'Abandono': '#e74c3c', 'Óbito': '#7f8c8d', 'Outros': '#95a5a6'}
CORES   = [PALETTE[g] for g in GRUPOS]

# Subconjunto binário para análise de regressão logística (Cura = 0, Abandono = 1)
df_bin = df[df['DESFECHO_GRUPO'].isin(['Cura', 'Abandono'])].copy()
df_bin['y'] = (df_bin['DESFECHO_GRUPO'] == 'Abandono').astype(int)

# Parâmetros globais de tipografia e estilo aplicados a todas as figuras do manuscrito
sns.set_theme(style='whitegrid', font_scale=1.6)
plt.rcParams.update({
    'font.family':      'DejaVu Sans',
    'font.size':        14,
    'axes.titlesize':   18,
    'axes.labelsize':   16,
    'xtick.labelsize':  14,
    'ytick.labelsize':  14,
    'legend.fontsize':  14,
    'figure.titlesize': 22,
})

w = 0.25  # largura das barras nos gráficos agrupados

# ═══════════════════════════════════════════════════════════════════════════════
# FIGURA 1 — Perfil comparado por desfecho (grade 2×2)
# ═══════════════════════════════════════════════════════════════════════════════
# Distribuições percentuais de raça/cor, sexo, faixa etária e fatores de
# vulnerabilidade social, estratificadas por desfecho clínico (Cura, Abandono, Óbito).

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(
    'Perfil Comparado: Cura vs Abandono vs Óbito\n(PE 2020–2024)',
    fontsize=20, fontweight='bold', y=1.02
)

# ── Raça/Cor ──────────────────────────────────────────────────────────────────
ax = axes[0, 0]
data      = (df[df['DESFECHO_GRUPO'].isin(GRUPOS)]
             .groupby(['DESFECHO_GRUPO', 'RACA_GRUPO']).size()
             .unstack(fill_value=0))
raca_order = [c for c in ['Negra', 'Branca', 'Indígena', 'Amarela', 'Ignorado'] if c in data.columns]
(data.div(data.sum(axis=1), axis=0) * 100).reindex(GRUPOS)[raca_order].plot(
    kind='bar', ax=ax, edgecolor='white', width=0.6)
ax.set_xticklabels(GRUPOS, rotation=0, fontsize=14)
ax.set(xlabel='', ylabel='%', ylim=(0, 100))
ax.set_title('Raça/Cor', fontsize=16, fontweight='bold')
ax.legend(title='', fontsize=12, loc='upper right')
ax.tick_params(axis='y', labelsize=12)

# ── Sexo ──────────────────────────────────────────────────────────────────────
ax = axes[0, 1]
data = (
    df[df['DESFECHO_GRUPO'].isin(GRUPOS)]
    .groupby(['DESFECHO_GRUPO', 'CS_SEXO']).size()
    .unstack(fill_value=0)
)
# Mantém apenas M e F, renomeia e reordena antes de plotar
data = data[['F', 'M']].rename(columns={'F': 'Feminino', 'M': 'Masculino'})
data_pct = (data.div(data.sum(axis=1), axis=0) * 100).reindex(GRUPOS)
data_pct.index.name = None
data_pct.plot(
    kind='bar', ax=ax,
    color=['#e91e63', '#3498db'],   # Feminino=rosa, Masculino=azul
    edgecolor='white', width=0.6
)
ax.set_xticklabels(GRUPOS, rotation=0, fontsize=14)
ax.set_ylabel('%', fontsize=13)
ax.set(ylim=(0, 100))
ax.set_title('Sexo', fontsize=16, fontweight='bold')
ax.legend(title='', fontsize=12, loc='upper right')
ax.tick_params(axis='y', labelsize=12)

# ── Faixa Etária ──────────────────────────────────────────────────────────────
ax = axes[1, 0]
data = (df[df['DESFECHO_GRUPO'].isin(GRUPOS)]
        .groupby(['DESFECHO_GRUPO', 'FAIXA_ETARIA'], observed=True).size()
        .unstack(fill_value=0))
(data.div(data.sum(axis=1), axis=0) * 100).reindex(GRUPOS).plot(
    kind='bar', ax=ax, edgecolor='white', width=0.6)
ax.set_xticklabels(GRUPOS, rotation=0, fontsize=14)
ax.set_xlabel('')
ax.set_title('Faixa Etária', fontsize=16, fontweight='bold')
ax.legend(title='', fontsize=12, loc='upper right')
ax.tick_params(axis='y', labelsize=12)

# ── Fatores de Vulnerabilidade ────────────────────────────────────────────────
ax = axes[1, 1]
FATORES_VULN = {
    'Situação\nde rua':  'RE_POP_RUA',
    'Alcoolismo':        'RE_AGRAVALCOO',
    'Uso de\ndrogas':    'RE_AGRAVDROGA',
    'Benef.\nsocial':    'RE_BENEF_GOV',
    'Baixa\nescolar.':   'RE_ESCOL',
}
x = np.arange(len(FATORES_VULN))
for i, (grupo, cor) in enumerate(zip(GRUPOS, CORES)):
    sub  = df[df['DESFECHO_GRUPO'] == grupo]
    vals = [sub[col].mean() * 100 for col in FATORES_VULN.values()]
    bars = ax.bar(x + i * w, vals, w, label=grupo, color=cor, edgecolor='white')
    for bar, v in zip(bars, vals):
        if v > 2:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                    f'{v:.0f}%', ha='center', va='bottom', fontsize=9, color='#333333')
ax.set_xticks(x + w)
ax.set_xticklabels(list(FATORES_VULN.keys()), fontsize=12)
ax.set_ylabel('%', fontsize=13)
ax.set_title('Fatores de Vulnerabilidade', fontsize=16, fontweight='bold')
ax.legend(fontsize=12)
ax.tick_params(axis='y', labelsize=12)

plt.tight_layout()
plt.savefig('fig1_perfil_comparado.png', dpi=200, bbox_inches='tight')
plt.close()
print("✓ fig1_perfil_comparado.png")

# ═══════════════════════════════════════════════════════════════════════════════
# BLOCO 6 — Análise Bivariada: Qui-quadrado + Odds Ratio (IC 95%)
# ═══════════════════════════════════════════════════════════════════════════════
# Para cada fator de vulnerabilidade, calcula-se a tabela de contingência 2×2,
# o teste qui-quadrado de Pearson (sem correção de Yates) e a OR bruta por
# regressão logística simples (método MLE). A variável desfecho é binária
# (Abandono = 1, Cura = 0), restrita ao subconjunto df_bin definido no Bloco 5.

FATORES_BIVARIADA = [
    ('RE_POP_RUA',    'Situação de rua'),
    ('RE_AGRAVALCOO', 'Alcoolismo'),
    ('RE_AGRAVDROGA', 'Uso de drogas'),
    ('RE_POP_LIBER',  'Privação de liberdade'),
    ('RE_BENEF_GOV',  'Beneficiário de prog. social'),
    ('RE_ESCOL',      'Baixa escolaridade'),
    ('RE_RACA',       'Raça vulnerabilizada (negro/pardo/indígena)'),
]

# Inclui sexo masculino como variável independente adicional
df_bin['SEXO_MASC'] = (df_bin['CS_SEXO'] == 'M').astype(int)
FATORES_BIVARIADA.append(('SEXO_MASC', 'Sexo masculino'))

resultados = []

for col, label in FATORES_BIVARIADA:
    sub = df_bin[[col, 'y']].dropna()
    sub[col] = sub[col].astype(int)

    # Tabela de contingência 2×2 (exposto/não-exposto × Abandono/Cura)
    tabela = pd.crosstab(sub[col], sub['y'])
    if tabela.shape != (2, 2):
        continue  # pula variáveis sem variação suficiente para o teste

    # Qui-quadrado de Pearson sem correção de continuidade
    chi2, p_valor, _, _ = chi2_contingency(tabela, correction=False)

    # OR bruta com IC 95% por regressão logística simples
    X = sm.add_constant(sub[col])
    modelo  = sm.Logit(sub['y'], X).fit(disp=0)
    or_val  = np.exp(modelo.params[col])
    ic_low  = np.exp(modelo.conf_int().loc[col, 0])
    ic_high = np.exp(modelo.conf_int().loc[col, 1])

    # Taxas de abandono estratificadas por exposição ao fator
    taxa_exp   = sub[sub[col] == 1]['y'].mean() * 100
    taxa_nexp  = sub[sub[col] == 0]['y'].mean() * 100
    n_exp      = int(sub[col].sum())

    resultados.append({
        'Fator':           label,
        'N exposto':       n_exp,
        'Abandono exposto (%)':     round(taxa_exp, 1),
        'Abandono não-exposto (%)': round(taxa_nexp, 1),
        'OR':      round(or_val, 2),
        'IC_low':  round(ic_low, 2),
        'IC_high': round(ic_high, 2),
        'p':       round(p_valor, 4),
        'p_str':   '< 0,001' if p_valor < 0.001 else f'{p_valor:.3f}',
    })

df_biv = pd.DataFrame(resultados)

# ── Tabela formatada para o artigo ───────────────────────────────────────────
print("\nTabela 1. Análise Bivariada — Fatores Associados ao Abandono do Tratamento de TB (PE 2020–2024)")
print(f"\n{'Fator':<45} {'N exp':>7}  {'Aban. exp.%':>11}  {'Aban. n-exp.%':>13}  {'OR':>6}  {'IC 95%':>14}  {'p':>8}")
print("─" * 110)
for _, r in df_biv.sort_values('OR', ascending=False).iterrows():
    ic_str = f"[{r['IC_low']:.2f}–{r['IC_high']:.2f}]"
    print(f"{r['Fator']:<45} {r['N exposto']:>7,}  "
          f"{r['Abandono exposto (%)']:>11.1f}  "
          f"{r['Abandono não-exposto (%)']:>13.1f}  "
          f"{r['OR']:>6.2f}  {ic_str:>14}  {r['p_str']:>8}")


Lendo dados de TB 2020–2024...
  TUBEBR20.csv: 86,160 linhas
  TUBEBR21.csv: 91,415 linhas
  TUBEBR22.csv: 103,382 linhas
  TUBEBR23.csv: 109,919 linhas
  TUBEBR24.csv: 113,012 linhas

[BLOCO 1] Total de registros (sem filtro): 503,888

[BLOCO 2] Registros residentes em PE:    33,688

          Distribuição por ano:
NU_ANO
2020    5389
2021    6027
2022    7035
2023    7449
2024    7441
2025     347

[BLOCO 3] Registros com ao menos 1 fator de vulnerabilidade: 29,270

Detalhamento por fator:
Fator                                               N       %
--------------------------------------------------------------
Baixa escolaridade (até fund. incompleto)      13,329   39.6%
Pessoas pretas, pardas ou indígenas            25,264   75.0%
Beneficiários de programa social                3,209    9.5%
Privados de liberdade                           5,055   15.0%
Em situação de rua                              1,057    3.1%
Imigrantes                                         70    0.2%
Alcool